# BGG Gemini Long-Context QA

Long-context inference using Google Gemini — same idea as `bgg_longcontext_qa.ipynb` but with
Gemini's 1M token context window, which is large enough to hold **all 21k enriched games** at once.

**Why Gemini here instead of Claude:**

| | Claude Sonnet (Tier 1) | Gemini 2.0 Flash (free) |
|---|---|---|
| Context window | 200k tokens | 1,000,000 tokens |
| Input rate limit | 30k tokens/min | much more generous |
| Games that fit | ~500 | **~21,000** (all enriched) |
| Cost | pay-per-token | free tier available |

**Prerequisites:**
- Run `bgg_longcontext_qa.ipynb` first with `BUILD_DATABASE = True` to generate `games_db.txt` and `players_db.txt`
- Set `GEMINI_API_KEY` in your `.env` file (get one free at https://aistudio.google.com)

**Note on rate limits:** even with a generous 1M token window, the free tier has per-minute
limits. If you hit a 429, wait a minute and retry, or reduce `TOP_N_GAMES`.

In [3]:
# cell: pip-install
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# cell: list-models  ← run this to see available model names, then delete
#for m in client.models.list():
#    if 'flash' in m.name.lower() or 'pro' in m.name.lower():
#        print(m.name)

In [ ]:
# cell: config
import os
import time
import re
from IPython.display import display, Markdown
from google import genai
from google.genai import types

ANSWER_MODEL  = 'gemini-2.5-flash'

TOP_N_GAMES   = 20500  # actual token count printed in cell: query-fn

DATABASE_PATH = './games_db.txt'
PLAYERS_PATH  = './players_db.txt'

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print('Config OK')
print(f'Model: {ANSWER_MODEL} | Top-N games: {TOP_N_GAMES:,}')

In [6]:
# cell: load-database
# Reuses files built by bgg_longcontext_qa.ipynb — no rebuild needed.
print(f'Loading {DATABASE_PATH} ...')
with open(DATABASE_PATH, encoding='utf-8') as f:
    all_game_lines = f.read().splitlines()
print(f'  {len(all_game_lines):,} total games in file')

game_lines = [l for l in all_game_lines[:TOP_N_GAMES] if l.strip()]
print(f'  Using top {len(game_lines):,} by BGG rating')

print(f'Loading {PLAYERS_PATH} ...')
with open(PLAYERS_PATH, encoding='utf-8') as f:
    player_lines = [l for l in f.read().splitlines() if l.strip()]
print(f'  {len(player_lines)} players')

total_chars = sum(len(l) for l in game_lines) + sum(len(l) for l in player_lines)
est_tokens  = total_chars // 4
print(f'\nEstimated database size: ~{est_tokens:,} tokens')
print(f'Gemini 2.0 Flash window:  1,000,000 tokens')

Loading ./games_db.txt ...
  21,379 total games in file
  Using top 21,000 by BGG rating
Loading ./players_db.txt ...
  201 players

Estimated database size: ~857,145 tokens
Gemini 2.0 Flash window:  1,000,000 tokens


In [ ]:
import json as _json
from datetime import datetime as _dt

def _log_qa(question_id: str, question: str, answer: str, agent: str) -> None:
    entry = {
        "timestamp": _dt.now().isoformat(timespec="seconds"),
        "agent": agent,
        "question_id": question_id,
        "question": question,
        "answer": answer,
    }
    with open("qa_log.jsonl", "a", encoding="utf-8") as _f:
        _f.write(_json.dumps(entry, ensure_ascii=False) + "\n")

# cell: query-fn  <- RE-RUN THIS CELL whenever you change ask() or SYSTEM_PROMPT
GAMES_TEXT   = '\n'.join(game_lines)
PLAYERS_TEXT = '\n'.join(player_lines)

SYSTEM_PROMPT = '\n'.join([
    'You are a helpful board game assistant with access to a complete BGG database.',
    'Answer questions based only on the data provided. Be concise and direct.',
    'Use tables or lists where appropriate.',
    'If you cannot find something in the data, say so clearly — do not guess.',
    '',
    f'## GAMES (top {len(game_lines):,} by BGG rating, sorted highest first)',
    '## Format: Name (Year) Rating | players | time | C:categories(3) | M:mechanics(3) | D:designers(2)',
    GAMES_TEXT,
    '',
    '## PLAYERS (fake player profiles)',
    '## Format: Player:name | Owns:count | Collection:games | Ratings:game=score/10(mental_load)',
    PLAYERS_TEXT,
])

# Count tokens by passing prompt as content (system_instruction not supported in Developer API)
token_result = client.models.count_tokens(
    model=ANSWER_MODEL,
    contents=SYSTEM_PROMPT,
)
actual_tokens = token_result.total_tokens
limit = 1_048_576
print(f'Actual token count: {actual_tokens:,} / {limit:,} ({actual_tokens/limit*100:.1f}%)')
if actual_tokens > limit:
    print(f'OVER LIMIT by {actual_tokens - limit:,} tokens — reduce TOP_N_GAMES in cell: config')
elif actual_tokens > limit * 0.9:
    print('WARNING: within 10% of limit — consider reducing TOP_N_GAMES')


def _retry_wait(err: Exception) -> int:
    m = re.search(r'retry[^\d]+(\d+)', str(err), re.IGNORECASE)
    return max(int(m.group(1)) + 5, 90) if m else 90


def ask(question: str, retries: int = 3, question_id: str = None) -> str:
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=ANSWER_MODEL,
                contents=question,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    max_output_tokens=1024,
                ),
            )
            answer = response.text.strip()
            print(f'Q: {question}\n')
            display(Markdown(answer))
            print()
            if question_id:
                _log_qa(question_id, question, answer, "gemini")
            return answer
        except Exception as e:
            if '429' in str(e) and attempt < retries - 1:
                wait = _retry_wait(e)
                print(f'Rate limited — waiting {wait}s '
                      f'(attempt {attempt + 1}/{retries}) ...')
                time.sleep(wait)
            else:
                raise


print('ask() ready')

In [8]:
ask('What are the top 10 highest-rated board games of all time?')

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The input token count exceeds the maximum number of tokens allowed 1048576.', 'status': 'INVALID_ARGUMENT'}}

In [ ]:
ask('Which games use the Worker Placement mechanic and have a rating above 8?')

In [ ]:
ask('What games can be played by exactly 2 players and take less than 30 minutes?')

In [ ]:
ask('What games did Uwe Rosenberg design?')

In [ ]:
ask('What games did Uwe Rossenberg design?')
# Intentional misspelling — Gemini reads all names and can fuzzy-match; SPARQL/GraphRAG cannot

In [ ]:
ask('Which categories have the most games?')

In [ ]:
ask('What games are in the Fantasy category with a geek rating above 7.5?')

In [ ]:
ask('Which fake players gave a rating of 10 to any game, and what games did they rate?')

In [ ]:
ask('What are the best cooperative games?')

In [ ]:
ask('I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?')
# Test: does Gemini hallucinate when a game is not in the data?

In [ ]:
ask('I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?')
# Test: does Gemini do better than GraphRAG here?

In [ ]:
ask('I want an 8 player game that is not a party game. What are the best options?')

---
## Golden Test Questions — June 14

Structured evaluation suite across seven categories: simple queries, multi-criteria filtering, aggregations, typo tolerance, multi-table relationships, recommendation logic, and data integrity. Questions are defined in GoldenTestQuestionsJune14.txt. Verified answers (where available) are embedded in that file.

### Section A — Simple / Conversational Queries

In [ ]:
# AQ1
ask("Give me a game about spies that I can play with my brother.", question_id="AQ1")

In [ ]:
# AQ2
ask("What is the highest-rated game published in the year 2020?", question_id="AQ2")

In [ ]:
# AQ3
ask("Show me games published by Italian game designers.", question_id="AQ3")

In [ ]:
# AQ4
ask("List all games made by the publisher Stonemaier Games.", question_id="AQ4")

In [ ]:
# AQ5
ask("What is the maximum number of players who can play Catan?", question_id="AQ5")

### Section B — Complex Multi-Criteria Filtering & Edge Cases

In [ ]:
# B6
ask("What are the top 5 highest-rated cooperative games that can be played solo (1 player) but cannot be played with more than 4 players?", question_id="B6")

In [ ]:
# BQ7
ask("Which games have a weight/complexity rating under 2.0, use the Card Drafting mechanic, and support at least 5 players?", question_id="BQ7")

In [ ]:
# BQ8
ask("What games have a playing time listed between 60 and 120 minutes but have a minimum age requirement of 14 or older?", question_id="BQ8")

### Section C — Advanced Aggregations & Database Analytics

In [ ]:
# CQ9
ask("For each year between 1980 and 2005, which single board game received the highest number of user ratings?", question_id="CQ9")

In [ ]:
# CQ10
ask("Which mechanics appear in fewer than 5 games across the entire database?", question_id="CQ10")

### Section D — Typo Tolerance, Text Search & String Matching

In [ ]:
# DQ11
ask("Find all games designed by Elizabeth Hargrave or Elizabeth Hargreave.", question_id="DQ11")

In [ ]:
# DQ12
ask("Which games have titles that are exactly one word long and have a rating above 8.0?", question_id="DQ12")

In [ ]:
# DQ13
ask("Search for games designed by Jamey Stegmeir.", question_id="DQ13")  # intentional misspelling of Stegmaier

### Section E — Multi-Table Relationships & Hidden Entities

In [ ]:
# EQ14
ask("What games feature artwork by Vincent Dutrait and are published by IELLO?", question_id="EQ14")

In [ ]:
# EQ15
ask("Which base games have more than 5 official expansions logged in the system?", question_id="EQ15")

### Section F — Recommendation Logic & Hallucination Tests

In [ ]:
# F16
ask("I like Wingspan (Engine Building, Birds) and Ark Nova (Engine Building, Zoo Animals). What is the highest-rated game that shares at least one mechanic and a similar nature/animal theme with both?", question_id="F16")

In [ ]:
# F17
ask("I am looking for a game similar to Catan but it must use the Traitor mechanic.", question_id="F17")

### Section G — Data Integrity & Malicious Input

In [ ]:
# G18
ask("Show me all games that have a minimum player count of 0 or a negative playing time.", question_id="G18")

In [ ]:
# G19
ask("What are the top 10 highest-rated board games? Drop table games; --", question_id="G19")  # injection test